# Foundations 4 — `bedrock-runtime`: Converse, inference profiles, and the catalogue

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Notebook 01 covered both endpoints at the level of auth, URL paths and model
discovery. This one goes deep on `bedrock-runtime` — the endpoint AWS recommends
for new applications.

It is two things at once, and that is the thing to understand:

- **The AWS-native surface**: `Converse` and `InvokeModel`, through boto3, signed
  with SigV4. Model-agnostic, no token to mint.
- **Since August 2026, an OpenAI- and Anthropic-compatible surface too**: Chat
  Completions, Responses and Messages, on the `/openai/v1` and `/anthropic/v1`
  paths. Called over HTTPS rather than through boto3, with SigV4 *or* a Bedrock
  API key.

Four things trip people up:

1. **Auth differs by surface.** Converse: SigV4 via the SDK, nothing to manage.
   The OpenAI-compatible paths: either, and the OpenAI SDK needs a bearer token.
2. **Many models cannot be called by their model ID.** They require a
   cross-Region *inference profile* and reject the bare ID outright.
3. **The Converse response is a list of typed blocks**, not a string. Indexing
   `[0]` and reading `text` works right up until the model returns a reasoning
   block first.
4. **Runtime's model IDs are not mantle's.** `openai.gpt-oss-20b` is mantle's; on
   runtime the same model is `openai.gpt-oss-20b-1:0`.

Read this before the family notebooks if you have not used Converse before.


### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `control_client` | a boto3 `bedrock` client (model and profile catalogues) |
| `converse` | one Converse call; returns `(text, response)` and **never raises** on a service error |
| `converse_reasoning` | the reasoning trace from a Converse response, or `""` |
| `converse_text` | concatenates the text blocks of a Converse response — safer than `content[0]` |
| `inference_profiles` | every inference-profile ID in a Region, cached |
| `resolve_runtime_id` | turns a model ID into the form Converse will accept, adding the `us.` profile prefix when one is required |
| `runtime_client` | a boto3 `bedrock-runtime` client (Converse, InvokeModel) |
| `runtime_models` | the serverless `bedrock-runtime` catalogue with modalities and inference types |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import sys

sys.path.insert(0, "../_shared")

import json

from bedrock import (
    control_client,
    converse,
    converse_reasoning,
    converse_text,
    inference_profiles,
    resolve_runtime_id,
    runtime_client,
    runtime_models,
)

REGION = "us-east-1"
CLAUDE = "anthropic.claude-sonnet-5"  # inference-profile only
NOVA = "amazon.nova-micro-v1"  # on-demand capable

runtime = runtime_client(REGION)
print("client:", type(runtime).__name__)
print("no bearer token needed - boto3 signs each request with SigV4")


client: BedrockRuntime
no bearer token needed - boto3 signs each request with SigV4


## 1. Auth: nothing to mint

On `bedrock-mantle` you mint a short-term bearer token that expires within 12
hours. On `bedrock-runtime` the SDK signs each request with your ambient
credentials, so there is no token lifecycle to manage at all. That is the single
biggest operational difference between the two endpoints.


In [2]:
text, response = converse(
    NOVA,
    [{"role": "user", "content": [{"text": "Reply with exactly: OK"}]}],
    max_tokens=16,
    region=REGION,
)
print("answer     :", text.strip())
print("stop reason:", response.get("stopReason"))
print("usage      :", response.get("usage"))


answer     : OK
stop reason: end_turn
usage      : {'inputTokens': 5, 'outputTokens': 2, 'totalTokens': 7}


## 2. The request shape

Converse normalises the request across providers, which is its main selling
point: the same call works for Nova, Claude and Llama. Three things differ from
the OpenAI shape:

- `content` is a **list of blocks**, not a string
- the token budget lives in `inferenceConfig`, not at the top level
- the system prompt is its own `system` parameter, not a message with
  `role: "system"`


In [3]:
raw = runtime.converse(
    modelId=resolve_runtime_id(NOVA, REGION),
    system=[{"text": "You are terse."}],
    messages=[
        {"role": "user", "content": [{"text": "Name one benefit of queues."}]},
        # A prior assistant turn goes here too, same block shape - that is how
        # you carry multi-turn state on Converse: you resend the history each
        # time. Note that this is a property of CONVERSE, not of the endpoint --
        # the Responses API on this same endpoint does keep server-side state
        # (§8 below).
    ],
    inferenceConfig={"maxTokens": 80, "temperature": 0.3},
)
print(json.dumps({k: v for k, v in raw.items() if k != "ResponseMetadata"},
                 indent=2, default=str)[:700])


{
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "Orderliness."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 10,
    "outputTokens": 4,
    "totalTokens": 14
  },
  "metrics": {
    "latencyMs": 364
  }
}


## 3. Never index `content[0]`

The response `content` list can contain `text`, `reasoningContent`, `toolUse` and
other block types, **in whatever order the model produced them**. A reasoning
model puts its trace first, so `content[0]["text"]` raises `KeyError`.

The cell below proves it with `moonshot.kimi-k2-thinking`, which reliably returns
a `reasoningContent` block ahead of its answer — and it budgets 2,500 tokens,
because this model can spend a smaller budget entirely on the trace and return no
text block at all. It is not only reasoning models,
though, and it is not stable per model: Claude Sonnet 5 returned a non-text first
block on one call and a text first block on the next while this notebook was
being written. So do not special-case the models you know about — walk the list
and select by key every time. `converse_text()` and `converse_reasoning()` in
`_shared/bedrock.py` do exactly that.


In [4]:
REASONER = "moonshot.kimi-k2-thinking"

text, response = converse(
    REASONER,
    [{"role": "user", "content": [{"text": "What is 17 * 23? Think it through."}]}],
    # Generous on purpose. At 600 tokens this model spends the entire budget on its
    # reasoning trace and returns NO text block, so converse_text() correctly
    # returns "" -- which made the "safe read" below look broken rather than safe.
    max_tokens=2500,
    region=REGION,
)

blocks = response.get("output", {}).get("message", {}).get("content", [])
print(f"{REASONER} returned {len(blocks)} content block(s):")
for i, block in enumerate(blocks):
    print(f"  content[{i}] -> {next(iter(block))}")

print("\nthe naive read:")
try:
    print("  content[0]['text'] =", blocks[0]["text"][:40])
except KeyError:
    print("  content[0]['text'] -> KeyError, because block 0 is the reasoning trace")

print("\nthe safe read:")
answer = converse_text(response).strip()
print("  converse_text()     :", answer[:70] or "(empty - budget exhausted)")
reasoning = converse_reasoning(response)
print("  converse_reasoning():", f"{len(reasoning)} chars" if reasoning else "(none)")
print(f"  stopReason          : {response.get('stopReason')}")
if not answer:
    print("  -> empty text with stopReason=max_tokens means the trace ate the")
    print("     budget. That is a budget problem, not a parsing problem.")

# Same request against Claude. Block ordering here varies between calls, which is
# the reason to never rely on it.
text2, response2 = converse(
    CLAUDE,
    [{"role": "user", "content": [{"text": "What is 17 * 23? Think it through."}]}],
    max_tokens=2500,
    region=REGION,
)
kinds = [
    next(iter(b))
    for b in response2.get("output", {}).get("message", {}).get("content", [])
]
print(f"\n{CLAUDE} returned blocks: {kinds}")
print("  -> may or may not lead with text; treat the order as undefined.")

moonshot.kimi-k2-thinking returned 2 content block(s):
  content[0] -> reasoningContent
  content[1] -> text

the naive read:
  content[0]['text'] -> KeyError, because block 0 is the reasoning trace

the safe read:
  converse_text()     : Let me solve this step by step.

**Method 1: Standard multiplication**
  converse_reasoning(): 998 chars
  stopReason          : end_turn



anthropic.claude-sonnet-5 returned blocks: ['text']
  -> may or may not lead with text; treat the order as undefined.


## 4. Inference profiles, and why a model ID can be rejected

Bedrock addresses a model on `bedrock-runtime` two ways:

    bare model ID          amazon.nova-micro-v1:0
    inference profile ID   us.amazon.nova-micro-v1:0

A profile routes your request across several Regions in one geography, which
raises availability and effective throughput. This is Cross-Region Inference
(CRIS).

The catch: models listed as `INFERENCE_PROFILE` only — which today includes
almost the whole Claude family — **reject the bare ID**. Models listed as
`ON_DEMAND` accept either form.


In [5]:
profile = f"us.{CLAUDE}"
detail = control_client(REGION).get_inference_profile(
    inferenceProfileIdentifier=profile
)
print(f"{detail['inferenceProfileId']}  ({detail.get('type')}, {detail.get('status')})")
print("routes your request to:")
for model in detail.get("models", []):
    arn = model["modelArn"]
    print(f"    {arn.split(':')[3]:<12} {arn.split('/')[-1]}")

print(f"\ntotal profiles in {REGION}: {len(inference_profiles(REGION))}")

print("\nwhat the bare ID does:")
try:
    runtime.converse(
        modelId=CLAUDE,
        messages=[{"role": "user", "content": [{"text": "Reply OK"}]}],
        inferenceConfig={"maxTokens": 16},
    )
    print("    accepted")
except Exception as exc:
    print(f"    {type(exc).__name__}: {str(exc)[-120:]}")

print("\nresolve_runtime_id() picks the right form for you:")
for model in (CLAUDE, NOVA):
    print(f"    {model:<28} -> {resolve_runtime_id(model, REGION)}")


us.anthropic.claude-sonnet-5  (SYSTEM_DEFINED, ACTIVE)
routes your request to:
    us-east-1    anthropic.claude-sonnet-5
    us-east-2    anthropic.claude-sonnet-5
    us-west-2    anthropic.claude-sonnet-5

total profiles in us-east-1: 71

what the bare ID does:


    ValidationException: mand throughput isn’t supported. Retry your request with the ID or ARN of an inference profile that contains this model.

resolve_runtime_id() picks the right form for you:
    anthropic.claude-sonnet-5    -> us.anthropic.claude-sonnet-5
    amazon.nova-micro-v1         -> us.amazon.nova-micro-v1:0


## 5. Reading the catalogue

`ListFoundationModels` is how you answer "what can I call, and how" without
guessing. Two fields matter most:

- `inputModalities` / `outputModalities` — what the model consumes and produces
- `inferenceTypesSupported` — `ON_DEMAND`, `INFERENCE_PROFILE`, or `PROVISIONED`

A model with only `INFERENCE_PROFILE` needs the `us.` form from section 4. A
model with only `PROVISIONED` needs a purchased throughput commitment and will
refuse on-demand calls entirely.


In [6]:
catalogue = runtime_models(REGION)
print(f"{len(catalogue)} catalogue entries in {REGION}\n")

import collections

by_type = collections.Counter()
for entry in catalogue.values():
    if "ON_DEMAND" in entry["infer"]:
        by_type["ON_DEMAND (bare ID works)"] += 1
    elif "INFERENCE_PROFILE" in entry["infer"]:
        by_type["INFERENCE_PROFILE only (us. required)"] += 1
    else:
        by_type["PROVISIONED only"] += 1
for label, count in by_type.most_common():
    print(f"  {count:>4}  {label}")

print("\nmodels that return text and accept images, on-demand:")
shown = 0
for key, entry in sorted(catalogue.items()):
    if entry["out"] == {"TEXT"} and "IMAGE" in entry["in"] and "ON_DEMAND" in entry["infer"]:
        print(f"    {key}")
        shown += 1
        if shown == 6:
            print("    ...")
            break


102 catalogue entries in us-east-1

    61  ON_DEMAND (bare ID works)
    41  INFERENCE_PROFILE only (us. required)

models that return text and accept images, on-demand:
    amazon.nova-lite-v1
    amazon.nova-pro-v1
    anthropic.claude-3-haiku-20240307-v1
    google.gemma-3-12b-it
    google.gemma-3-27b-it
    google.gemma-3-4b-it
    ...


## 6. Streaming, and the lower-level escape hatch

`converse_stream` gives you incremental output with the same normalised shape.
`invoke_model` is the raw passthrough: you send the provider's own JSON body and
get theirs back. Reach for `invoke_model` only when a provider exposes something
Converse has not normalised yet — otherwise Converse is less code and portable.


In [7]:
stream = runtime.converse_stream(
    modelId=resolve_runtime_id(NOVA, REGION),
    messages=[{"role": "user", "content": [{"text": "Count to five."}]}],
    inferenceConfig={"maxTokens": 60},
)
events = collections.Counter()
pieces = []
try:
    for event in stream["stream"]:
        kind = next(iter(event))
        events[kind] += 1
        if kind == "contentBlockDelta":
            pieces.append(event[kind]["delta"].get("text", ""))
except Exception as exc:
    # A stream can fail AFTER delivering part of the answer: a mid-stream
    # 5xx is not rare, and it has happened while building these notebooks.
    # Report what arrived instead of losing it - production code has to
    # decide whether a partial answer is usable or the call must be retried.
    print(f"\n[stream interrupted: {type(exc).__name__}]")
print("event types:", dict(events))
print("assembled  :", "".join(pieces).strip()[:90])

# invoke_model: the provider's own schema, not Converse's.
body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 24,
    "messages": [{"role": "user", "content": "Reply with exactly: OK"}],
}
raw = runtime.invoke_model(
    modelId=resolve_runtime_id(CLAUDE, REGION), body=json.dumps(body)
)
payload = json.loads(raw["body"].read())
print("\ninvoke_model top-level keys:", sorted(payload.keys()))
# Same discipline as section 3: select the text block, do not index [0].
answer = "".join(b["text"] for b in payload["content"] if b.get("type") == "text")
print("text:", answer.strip()[:60])


event types: {'messageStart': 1, 'contentBlockDelta': 16, 'contentBlockStop': 1, 'messageStop': 1, 'metadata': 1}
assembled  : Sure, let's count to five:

1. One
2. Two
3. Three
4. Four
5. Five

There we go! We've cou



invoke_model top-level keys: ['content', 'id', 'model', 'role', 'stop_details', 'stop_reason', 'stop_sequence', 'type', 'usage']
text: OK


## 7. Choosing an endpoint

AWS's guidance is now explicit: *"For new applications, we recommend the
`bedrock-runtime` endpoint."* So the question is no longer "which endpoint" but
"do I have a reason to add `bedrock-mantle`". Ask in this order:

1. **Is the model on `bedrock-runtime` at all?** Gemma 4, GPT-5.4, GPT-5.5, Grok
   4.3, DeepSeek v3.1 and GLM 4.6 are mantle-only today. That decides it. Do not
   trust that list — `endpoints_for()` reads both catalogues live, and §7b prints
   the current split.
2. **Do you need something only `bedrock-mantle` has?** Server-side or
   pre-configured tool use including web search; asynchronous inference with
   `background=true`; Projects or Workspaces for per-application cost attribution.
   Those are mantle-only, and they are the honest reasons to use it.
3. **Do you need something only `bedrock-runtime` has?** Guardrails, intelligent
   prompt routing, cross-Region inference, Provisioned Throughput, batch
   inference, `InvokeModel` for non-text modalities.
4. **Neither?** Use `bedrock-runtime`. That is the recommendation, and it is where
   the account-level controls you already use — invocation logging, CloudWatch
   metrics, Cost Explorer attribution — apply to the OpenAI-shaped calls too.

Pricing does not enter into it: per-token pricing for the same model is identical
on both endpoints. And both can be used from one application — choose per use
case.

Then, having chosen the endpoint, choose the API:

| You want | Use |
|---|---|
| one interface across every model | **Converse** |
| raw provider JSON, or a non-text modality | **InvokeModel** |
| to move existing OpenAI code with a base-URL change | **Chat Completions** or **Responses** |
| server-side conversation state, reasoning items, tool loops | **Responses** |
| to move existing Anthropic code | **Messages** |

### 7b. The split, printed live

Two catalogues, two naming conventions. This cell reconciles them so you can see
which models are actually mantle-only rather than merely *named differently*.

That distinction matters: comparing the two catalogues on exact model IDs makes
`openai.gpt-oss-120b` look mantle-only, when its runtime twin is
`openai.gpt-oss-120b-1:0`. `runtime_id_for()` normalises the four ways the names
differ — version suffix, `-v1:0` suffix, provider prefix (`moonshotai.` vs
`moonshot.`), and a trailing `-instruct`.

In [8]:
from bedrock import list_models, runtime_id_for

mantle_ids = sorted(list_models(REGION))
mapped = {m: runtime_id_for(m, REGION) for m in mantle_ids}

mantle_only = [m for m, r in mapped.items() if r is None]
renamed = {m: r for m, r in mapped.items() if r and r != m}
same = [m for m, r in mapped.items() if r == m]

print(f"{len(mantle_ids)} models on bedrock-mantle in {REGION}")
print(f"  {len(same):>3} reachable on runtime under the SAME id")
print(f"  {len(renamed):>3} reachable under a DIFFERENT id")
print(f"  {len(mantle_only):>3} not on runtime at all")

print("\nnot on bedrock-runtime -- these are the real reasons to use mantle:")
for m in mantle_only:
    print(f"    {m}")

print("\nrenamed (a sample; this is the class of bug that wastes an afternoon):")
for m, r in list(renamed.items())[:8]:
    print(f"    {m:36} -> {r}")
if len(renamed) > 8:
    print(f"    ... and {len(renamed) - 8} more")

# And the reverse direction: runtime carries families mantle never had.
runtime_only_families = sorted(
    {k.split(".")[0] for k in runtime_models(REGION)}
    - {m.split(".")[0] for m in mantle_ids}
)
print(f"\nfamilies only on runtime: {runtime_only_families}")

55 models on bedrock-mantle in us-east-1
   27 reachable on runtime under the SAME id
   16 reachable under a DIFFERENT id
   12 not on runtime at all

not on bedrock-runtime -- these are the real reasons to use mantle:
    deepseek.v3.1
    google.gemma-4-26b-a4b
    google.gemma-4-31b
    google.gemma-4-e2b
    openai.gpt-5.4
    openai.gpt-5.4-2026-03-05
    openai.gpt-5.5
    openai.gpt-5.5-2026-04-23
    qwen.qwen3-235b-a22b-2507
    qwen.qwen3-coder-480b-a35b-instruct
    xai.grok-4.3
    zai.glm-4.6

renamed (a sample; this is the class of bug that wastes an afternoon):
    anthropic.claude-fable-5             -> us.anthropic.claude-fable-5
    anthropic.claude-haiku-4-5           -> us.anthropic.claude-haiku-4-5-20251001-v1:0
    anthropic.claude-opus-4-7            -> us.anthropic.claude-opus-4-7
    anthropic.claude-opus-4-8            -> us.anthropic.claude-opus-4-8
    anthropic.claude-opus-5              -> us.anthropic.claude-opus-5
    anthropic.claude-sonnet-5          

## 8. The OpenAI- and Anthropic-compatible APIs, on `bedrock-runtime`

This is what changed in August 2026, and it is the reason the recommendation
moved. The same endpoint that serves Converse also serves:

| Path | API |
|---|---|
| `/openai/v1/chat/completions` | OpenAI Chat Completions |
| `/openai/v1/responses` | OpenAI Responses |
| `/anthropic/v1/messages` | Anthropic Messages |

Three practical notes before the code:

- **These paths are not in boto3.** You call them over HTTPS. `runtime_post()` in
  `_shared/bedrock.py` does it with a bearer token; SigV4 works too.
- **There is no `/v1` inference surface here.** Asking for one returns HTTP **200**
  with a Coral `UnknownOperationException` — `../00-foundations/01` §2b. Use
  `ok(status, body)` rather than `status == 200`.
- **Coverage is uneven, and narrower than Converse.** The cell measures it.

In [9]:
from bedrock import ok, runtime_post

CANDIDATES = [
    "openai.gpt-oss-120b",
    "openai.gpt-5.6-sol",
    "xai.grok-4.6",
    "anthropic.claude-opus-5",
    "qwen.qwen3-32b",
    "zai.glm-5",
    "amazon.nova-lite-v1",
    "meta.llama4-maverick-17b-instruct-v1",
]
AV = {"anthropic-version": "2023-06-01"}


def surface_check(model_id: str) -> dict:
    """Chat / Responses / Messages / Converse for one model on bedrock-runtime."""
    rid = runtime_id_for(model_id, REGION)
    row = {"asked": model_id, "runtime_id": rid or "-"}
    if rid is None:
        return {**row, "chat": "n/a", "responses": "n/a", "messages": "n/a",
                "converse": "n/a"}

    is_claude = "anthropic." in rid

    if is_claude:
        row["chat"] = row["responses"] = "-"
        code_, data = runtime_post(
            "/anthropic/v1/messages",
            {"model": rid, "max_tokens": 16,
             "messages": [{"role": "user", "content": "Hi"}]},
            region=REGION, headers=AV, attempts=1, timeout=90)
        row["messages"] = "ok" if ok(code_, data) else str(code_)
    else:
        row["messages"] = "-"
        code_, data = runtime_post(
            "/openai/v1/responses",
            {"model": rid, "input": "Hi", "max_output_tokens": 2000},
            region=REGION, attempts=1, timeout=120)
        row["responses"] = "ok" if ok(code_, data) else str(code_)
        # gpt-5.x and Grok want max_completion_tokens; the rest take max_tokens.
        for field in ("max_completion_tokens", "max_tokens"):
            code_, data = runtime_post(
                "/openai/v1/chat/completions",
                {"model": rid, "messages": [{"role": "user", "content": "Hi"}],
                 field: 2000},
                region=REGION, attempts=1, timeout=120)
            if ok(code_, data):
                break
        row["chat"] = "ok" if ok(code_, data) else str(code_)

    _, response = converse(
        rid, [{"role": "user", "content": [{"text": "Hi"}]}],
        max_tokens=512, region=REGION, resolve=False)
    row["converse"] = "err" if response.get("error") else "ok"
    return row

print(f"{'asked for':38} {'runtime id':32} {'Chat':>6} {'Resp':>6} {'Msg':>6} {'Conv':>6}")
print("-" * 100)
rows = [surface_check(m) for m in CANDIDATES]
for r in rows:
    print(f"{r['asked']:38} {r['runtime_id']:32} {r['chat']:>6} "
          f"{r['responses']:>6} {r['messages']:>6} {r['converse']:>6}")

counts = {
    key: sum(1 for r in rows if r[key] == "ok")
    for key in ("chat", "responses", "messages", "converse")
}
print(f"\n=> of {len(rows)} models: " + ", ".join(f"{k} {v}" for k, v in counts.items()))
print("   Converse is the widest surface on this endpoint by a long way. The")
print("   OpenAI-compatible paths are for moving existing code, not for reach.")

asked for                              runtime id                         Chat   Resp    Msg   Conv
----------------------------------------------------------------------------------------------------


openai.gpt-oss-120b                    openai.gpt-oss-120b-1:0              ok    404      -     ok
openai.gpt-5.6-sol                     us.openai.gpt-5.6-sol                ok     ok      -     ok
xai.grok-4.6                           us.xai.grok-4.6                      ok     ok      -     ok
anthropic.claude-opus-5                us.anthropic.claude-opus-5            -      -     ok     ok
qwen.qwen3-32b                         qwen.qwen3-32b-v1:0                  ok    404      -     ok
zai.glm-5                              zai.glm-5                            ok    400      -     ok
amazon.nova-lite-v1                    amazon.nova-lite-v1:0:24k           404    404      -    err
meta.llama4-maverick-17b-instruct-v1   us.meta.llama4-maverick-17b-instruct-v1:0    404    404      -     ok

=> of 8 models: chat 5, responses 2, messages 1, converse 7
   Converse is the widest surface on this endpoint by a long way. The
   OpenAI-compatible paths are for moving existing code, not

### 8b. Server-side conversation state

The Responses API on `bedrock-runtime` defaults to `store=true` and accepts
`previous_response_id`, so the service holds the history. This is the capability an
earlier version of this notebook said the endpoint did not have.

One caveat from the AWS docs that affects design: *"A stored response belongs to
the AWS Region that served it."* Retrieving, cancelling, deleting, or continuing
with `previous_response_id` all go to that Region. A `global.` profile does not
tell you which Region that was — so use the `us.` geo profile if you need state
pinned to a known geography.

In [10]:
STATE_MODEL = runtime_id_for("openai.gpt-5.6-sol", REGION)
print("using", STATE_MODEL)

code_, first = runtime_post(
    "/openai/v1/responses",
    {"model": STATE_MODEL, "input": "My badge number is 8812.", "store": True,
     "max_output_tokens": 2000},
    region=REGION, attempts=1, timeout=180)

if not ok(code_, first):
    print(f"turn 1 -> {code_}: {json.dumps(first)[:150]}")
else:
    print("store echoed:", first.get("store"))
    for label, extra in (
        ("with previous_response_id", {"previous_response_id": first["id"]}),
        ("without it (control)", {}),
    ):
        code_, reply = runtime_post(
            "/openai/v1/responses",
            {"model": STATE_MODEL, "input": "What is my badge number? Digits only.",
             "max_output_tokens": 2000, **extra},
            region=REGION, attempts=1, timeout=180)
        if ok(code_, reply):
            answer = "".join(
                p.get("text", "")
                for item in (reply.get("output") or [])
                for p in (item.get("content") or [])
                if isinstance(p, dict)
            ).strip()
            print(f"  {label:26} -> {answer[:70]!r}")
        else:
            print(f"  {label:26} -> {code_}")

    print("\n=> The control is the point: without the link the model cannot know the")
    print("   number, so the linked answer is evidence of server-side state rather")
    print("   than of a lucky guess. Always pair a state demo with its control.")

# background=true is the one Responses feature this endpoint does not have.
code_, data = runtime_post(
    "/openai/v1/responses",
    {"model": STATE_MODEL, "input": "Hi", "background": True,
     "max_output_tokens": 2000},
    region=REGION, attempts=1, timeout=120)
print(f"\nbackground=true on runtime -> {code_}: "
      f"{(data.get('error') or {}).get('message', '')[:70]}")
print("=> Asynchronous inference stays on bedrock-mantle.")

using us.openai.gpt-5.6-sol


store echoed: True


  with previous_response_id  -> '8812'


  without it (control)       -> '0'

=> The control is the point: without the link the model cannot know the
   number, so the linked answer is evidence of server-side state rather
   than of a lucky guess. Always pair a state demo with its control.



background=true on runtime -> 400: The background parameter is not supported.
=> Asynchronous inference stays on bedrock-mantle.


## Takeaways

- `bedrock-runtime` is the endpoint AWS recommends for new applications, and it
  serves five APIs: Converse, InvokeModel, Chat Completions, Responses, Messages.
- Converse needs no token minting; SigV4 does it. The OpenAI-compatible paths take
  SigV4 **or** a Bedrock API key, which is what the OpenAI SDK needs.
- Converse `content` is an ordered list of typed blocks. Select by key, never index
  `[0]` — the same model can put a reasoning block first on one call and not the
  next. Responses `output[]` behaves the same way.
- `INFERENCE_PROFILE`-only models reject the bare ID. Learn the error text; most of
  the Claude family, the GPT-5.6 family and Grok 4.6 produce it.
- Read `inferenceTypesSupported` before writing the call, not after debugging it.
- Prefer Converse over `invoke_model` unless you need something unnormalised, and
  prefer it over the OpenAI-compatible paths unless you are moving existing code —
  its model coverage on this endpoint is much wider.
- **Runtime model IDs are not mantle model IDs.** Resolve with `runtime_id_for()`;
  reusing a mantle ID gives *"The provided model identifier is invalid"*, which
  reads like a missing model rather than a missing translation.
- **A body containing `UnknownOperation` is a failure whatever the status says.**
  This endpoint returns HTTP 200 for a path it does not serve. Use
  `ok(status, body)`.
- Budget for the reasoning trace. An empty `text` block with
  `stopReason: max_tokens` is a budget problem, not a parsing problem — and it is
  indistinguishable from a broken parser if you do not print the stop reason.
- Server-side conversation state exists here: Responses defaults to `store=true`
  and accepts `previous_response_id`. Asynchronous inference (`background=true`)
  does not — that stays on `bedrock-mantle`.
- Guardrails attach here, not on `bedrock-mantle`: `guardrailConfig` on Converse,
  `guardrailIdentifier` on `invoke_model`. **Which OpenAI-shaped surfaces honour a
  guardrail header differs between them**, and one accepts it and does nothing —
  `../99-cross-cutting/03` §9b measures each.